In [1]:
from importlib import reload

from prompt_utils import build_prompt
from data_utils import create_submission, parse_predictions, create_comparison_df, read_dataset
from semevalpolar.llm import custom_rules
from semevalpolar.llm.main import test_run, create_gen, create_response
from tqdm import tqdm
from semevalpolar.utils import get_project_root
import pandas as pd
from custom_rules import apply_rules

import os
import ast

D:\projects\mldl\semevalpolar


In [2]:
batch_size = 10
data_path = os.path.join(get_project_root(), 'data', 'relabelling', 'eng', 'eng.csv')
gen = create_gen(data_path, batch_size=batch_size, randomize=True)
generator_list = list(gen)

In [3]:
combined_df = pd.concat(generator_list, ignore_index=True)

In [4]:
predictions = []
usages = []

In [5]:
for batch in tqdm(generator_list[:2]):
    response = test_run(batch, prompt_path="prompt-three-classes.txt")
    predictions.append(parse_predictions(response.output_text))
    usages.append(response.usage)

100%|██████████| 2/2 [00:05<00:00,  2.81s/it]


In [30]:
flat = [x for sub in predictions for x in sub]

In [33]:
combined_df.insert(2, "llm_polarization", pd.Series(flat).reset_index(drop=True))

In [34]:
combined_df

,id,text,llm_polarization,polarization_split,polarization
0,eng_725376b7deb2b678ba708f5c9e86c841,Imagine blaming this on Israel instead of Hamas,1.0,NaN,1
1,eng_eb56499c0778a9424138d6bffba09bce,Blinken is the Butcher of Blinken ButcherOfGaz...,0.0,NaN,1
2,eng_a7beb0cb3b45eb18195c4558a1fcbc03,"And as far as your gun control, Chicago knows ...",1.0,NaN,0
3,eng_fdc01502f73b3de226e90eacc233b1b2,All the spoilers are on Fox News daily.,0.0,NaN,0
4,eng_b48d32dd24061710af1d76d14e05a2f9,Are red states stealing from California?,1.0,NaN,1
...,...,...,...,...,...
3217,eng_06a4b4c509635bf4175a74e7556d0cc5,Kamala Harris will be POTUS. Amendment 14 Sect...,NaN,NaN,0
3218,eng_687668a8c953f14634d41c47058c6838,The West Bank got too many police,NaN,NaN,0
3219,eng_763fc6b6bf1158ab7ded46e164394902,You work for the IDF?,NaN,NaN,0
3220,eng_83294dfc9578e7f0a00c6ff2213d720c,They are terrified of multiculturalism.,NaN,NaN,0


In [ ]:
ground_truths = []

In [ ]:
for i in range(len(flat) // 10):
    for x in generator_list[i]["polarization"]:
        ground_truths.append(x)

In [ ]:
texts = []

for i in range(len(predictions)):
    for j in range(batch_size):
        texts.append(generator_list[i]["text"].iloc[j])

In [ ]:
from sklearn.metrics import confusion_matrix

comparison = create_comparison_df(flat, ground_truths, texts)
cm = confusion_matrix(comparison["Ground Truth"], comparison["Predicted"], labels=[0, 1])
cm

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(comparison["Ground Truth"], comparison["Predicted"])

In [ ]:
comparison

In [ ]:
correct = comparison[comparison["Predicted"] == comparison["Ground Truth"]]
correct

In [ ]:
wrong = comparison[comparison["Ground Truth"] != comparison["Predicted"]]
wrong[wrong["Predicted"] == 1]

In [ ]:
submission = create_submission(df, flat)

In [ ]:
submission.to_csv(os.path.join(get_project_root(), 'predictions', 'subtask_1', 'pred_hin.csv'), index=False)